<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# C18's Test, Piloted on Its Controls

A pilot of a test, not of the hunch. C18's plan was parked twice: its first model, run for a fixed horizon, would have
refuted the hunch by construction (journal of 2026-10-14, *C18's Model, Piloted*), and the width test the pilot led to
cannot tell a continuous transition from a smooth change (journal of 2026-10-15, *C18's Plan, Attempted Again*). That
entry proposed a test taken below and above a threshold instead, with controls. This notebook fixes that test's statistic
and margins before any data, then runs it on three controls whose answer is known, to show that it reads a threshold
where one is and none where none is. It runs none of the spreading models the plan will judge, so the plan's margins are
set on the controls alone.

**The test**, for a model with a mean degree `d` to sweep.

- *Stage 1, locate*: at N = 10 000, sweep `d` from 0.25 to 12 in steps of 0.25, 50 draws each; the located threshold
  `d_hat` is the degree at which the mean reached fraction first passes 1%, by linear interpolation between grid points
  (the grid's first point if it already passes 1%; a model whose sweep never passes it is tested at the grid's ends,
  0.25 and 12, instead).
- *Stage 2, test*: at `d_hat / 2` and `2 d_hat`, 200 draws at each of N = 100, 1000 and 10 000. The statistic is the
  ratio of the mean reached fraction at N = 100 to that at N = 10 000, with a 99% percentile bootstrap interval over
  draws (10 000 resamples, the two sizes resampled independently). The reached fraction is the share of the other
  N - 1 units reached. A draw whose seed has no out-edge is replaced and the replacements counted - the conditioning
  [#60](https://github.com/ai-quokka-wannabe/iso-lab/pull/60)'s review suggested, on the graph rather than on the outcome.
- *Margins, fixed before the run*: below the threshold, a ratio whose interval lies wholly above 10 reads **finite**
  (the mark of a threshold: a reach that stays finite while N grows, so a fraction falling about as 1/N, a hundredfold
  between these sizes), and one wholly below 3 reads **extensive**; above the threshold, an interval wholly below 2
  reads **holding** and one wholly above 5 reads **falling**. A **cliff** is finite below and holding above; **no
  cliff** is extensive below or falling above; anything else is **inconclusive**.
- *Descriptive, not decisive*: the slope of log fraction against log N over the three sizes (-1 for 1/N), and the
  paper's own signature - the coefficient of variation of the reached fraction over draws along the stage-1 sweep,
  which Shrager, Hogg and Huberman 1987 ([doi:10.1126/science.236.4805.1092](https://doi.org/10.1126/science.236.4805.1092))
  used to place their transition at N = 100, where the mean curve alone was too smooth to show it.

**The controls.**

1. **Giant component**: the seed's out-component in the directed random graph the pilot used (Poisson(dN) edges with
   uniform ends, self-loops dropped, repeated edges kept), whose continuous transition is at `d` = 1. A threshold with
   no jump in the limit: it must read *cliff*.
2. **Jump**: the same graph, but below `d` = 2.5 the seed reaches its out-neighbours only and from 2.5 on its whole
   out-component - a constructed first-order change, shaped like what the pure model does in the limit (journal of
   2026-10-15). It must read *cliff* too.
3. **Smooth**: each other unit reached independently with probability 1 / (1 + exp(-(d - 2.5) / 0.5)) - no graph and
   no threshold, so its fraction does not depend on N. It must read *no cliff*. It has no seed degree, so no draw of it
   is replaced.

**What decides.** The pilot passes if all three read as they must. If any does not, the statistic or its margins are
wrong, and the plan is not written with them. Harness checks come first: at `d` = 0 nothing beyond the seed is
reached, and on a complete graph everything is.


In [1]:
import os

import numpy as np

# nbconvert runs a notebook from its own directory; the figures live under the repository's root.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

SIZES = [100, 1000, 10000]
GRID = np.arange(0.25, 12.01, 0.25)
LOCATE_DRAWS, TEST_DRAWS, BOOT, ALPHA = 50, 200, 10000, 0.01
LEVEL = 0.01
MARGINS = {"finite below": 10.0, "extensive below": 3.0, "holding above": 2.0, "falling above": 5.0}
JUMP_AT, SMOOTH_MID, SMOOTH_WIDTH = 2.5, 2.5, 0.5
CONTROLS = ["giant component", "jump", "smooth"]
MUST_READ = {"giant component": "cliff", "jump": "cliff", "smooth": "no cliff"}


def network(n, d, rng):
    """The pilot's directed random graph: Poisson(d n) edges with uniform ends, self-loops dropped, repeats kept."""
    m = rng.poisson(d * n)
    src, dst = rng.integers(0, n, m), rng.integers(0, n, m)
    keep = src != dst
    return src[keep], dst[keep]


def out_component(n, src, dst, seed=0):
    """A mask of the units the seed reaches along directed edges, the seed included."""
    reached = np.zeros(n, dtype=bool)
    reached[seed] = True
    frontier = np.array([seed])
    while frontier.size:
        on = np.zeros(n, dtype=bool)
        on[frontier] = True
        new = dst[on[src]]
        new = np.unique(new[~reached[new]])
        reached[new] = True
        frontier = new
    return reached


def reach(control, n, d, rng):
    """The fraction of the other n - 1 units reached; None when the seed has no out-edge, so the draw is replaced."""
    if control == "smooth":
        p = 1.0 / (1.0 + np.exp(-(d - SMOOTH_MID) / SMOOTH_WIDTH))
        return rng.binomial(n - 1, p) / (n - 1)
    src, dst = network(n, d, rng)
    out = dst[src == 0]
    if out.size == 0:
        return None
    if control == "jump" and d < JUMP_AT:
        return np.unique(out).size / (n - 1)
    return (out_component(n, src, dst).sum() - 1) / (n - 1)


def draws(control, n, d, rng, k):
    """k eligible draws, and how many were replaced because the seed had no out-edge."""
    values, replaced = [], 0
    while len(values) < k:
        v = reach(control, n, d, rng)
        if v is None:
            replaced += 1
        else:
            values.append(v)
    return np.array(values), replaced


# Harness checks: nothing beyond the seed at d = 0; everything on a complete graph.
check = np.random.default_rng(0)
assert out_component(100, *network(100, 0.0, check)).sum() == 1
complete = np.array([(i, j) for i in range(100) for j in range(100) if i != j]).T
assert out_component(100, complete[0], complete[1]).sum() == 100
print("numpy", np.__version__, "- harness checks passed; generator seeds [20261031, 1] locate, [20261031, 2] test, [20261031, 3] bootstrap")


numpy 2.5.4 - harness checks passed; generator seeds [20261031, 1] locate, [20261031, 2] test, [20261031, 3] bootstrap


In [2]:
rng1 = np.random.default_rng([20261031, 1])
located = {}
print(f"stage 1: N = 10000, {LOCATE_DRAWS} draws per degree, grid {GRID[0]} to {GRID[-1]} step {GRID[1] - GRID[0]}")
print("control           d_hat   first grid d at or past 1%   CV peak at d   replaced draws over the sweep")
for control in CONTROLS:
    means, cvs, replaced_total = [], [], 0
    for d in GRID:
        v, replaced = draws(control, 10000, d, rng1, LOCATE_DRAWS)
        replaced_total += replaced
        means.append(v.mean())
        cvs.append(v.std(ddof=1) / v.mean() if v.mean() > 0 else np.nan)
    means, cvs = np.array(means), np.array(cvs)
    i = int(np.argmax(means >= LEVEL))
    if means[i] < LEVEL:
        d_hat = None
    elif i == 0:
        d_hat = float(GRID[0])
    else:
        d_hat = float(GRID[i - 1] + (LEVEL - means[i - 1]) * (GRID[i] - GRID[i - 1]) / (means[i] - means[i - 1]))
    located[control] = {"d_hat": d_hat, "means": means, "cvs": cvs}
    peak = GRID[int(np.nanargmax(cvs))]
    shown = "none" if d_hat is None else f"{d_hat:.3f}"
    print(f"{control:16s}  {shown:>6}   {GRID[i]:>6.2f}                       {peak:>5.2f}          {replaced_total}")


stage 1: N = 10000, 50 draws per degree, grid 0.25 to 12.0 step 0.25
control           d_hat   first grid d at or past 1%   CV peak at d   replaced draws over the sweep


giant component    1.005     1.25                        1.00          380


jump               2.253     2.50                        1.50          410
smooth             0.250     0.25                        0.25          0


In [3]:
rng2 = np.random.default_rng([20261031, 2])
rng3 = np.random.default_rng([20261031, 3])


def ratio_interval(a, b, rng):
    """The ratio of means a / b, with a percentile bootstrap interval at level 1 - ALPHA, a and b resampled independently."""
    ia = rng.integers(0, a.size, (BOOT, a.size))
    ib = rng.integers(0, b.size, (BOOT, b.size))
    r = a[ia].mean(axis=1) / b[ib].mean(axis=1)
    return a.mean() / b.mean(), float(np.quantile(r, ALPHA / 2)), float(np.quantile(r, 1 - ALPHA / 2))


def reading(below, above):
    """The pre-registered reading from the two intervals (low, high)."""
    finite = below[0] > MARGINS["finite below"]
    extensive = below[1] < MARGINS["extensive below"]
    holding = above[1] < MARGINS["holding above"]
    falling = above[0] > MARGINS["falling above"]
    if finite and holding:
        return "cliff"
    if extensive or falling:
        return "no cliff"
    return "inconclusive"


results = {}
print(f"stage 2: {TEST_DRAWS} draws per point, {BOOT} bootstrap resamples, {100 * (1 - ALPHA):.0f}% intervals")
for control in CONTROLS:
    d_hat = located[control]["d_hat"]
    if d_hat is None:  # never passed 1%: the grid's ends stand in, and the reading says whether anything spreads at all
        points = (("below", float(GRID[0])), ("above", float(GRID[-1])))
    else:
        points = (("below", d_hat / 2), ("above", 2 * d_hat))
    rows = {}
    for side, d in points:
        samples = {n: draws(control, n, d, rng2, TEST_DRAWS) for n in SIZES}
        means = np.array([samples[n][0].mean() for n in SIZES])
        ratio, lo, hi = ratio_interval(samples[100][0], samples[10000][0], rng3)
        slope = float(np.polyfit(np.log(SIZES), np.log(means), 1)[0])
        rows[side] = {"d": d, "means": means, "ratio": ratio, "lo": lo, "hi": hi, "slope": slope,
                      "replaced": {n: samples[n][1] for n in SIZES}}
    verdict = reading((rows["below"]["lo"], rows["below"]["hi"]), (rows["above"]["lo"], rows["above"]["hi"]))
    results[control] = {"rows": rows, "verdict": verdict}
    print(f"\n{control}: d_hat = " + ("none, so the grid's ends stand in" if d_hat is None else f"{d_hat:.3f}"))
    for side in ("below", "above"):
        r = rows[side]
        m = ", ".join(f"{n}: {x:.5f}" for n, x in zip(SIZES, r["means"]))
        print(f"  {side} (d = {r['d']:.3f}): mean reached fraction by N {m}")
        print(f"    ratio N=100 / N=10000 = {r['ratio']:.2f} [{r['lo']:.2f}, {r['hi']:.2f}]; slope of log fraction on log N {r['slope']:.2f};"
              f" draws replaced {r['replaced']}")
    print(f"  reads: {verdict}; must read: {MUST_READ[control]} -> {'as it must' if verdict == MUST_READ[control] else 'NOT as it must'}")

passed = all(results[c]["verdict"] == MUST_READ[c] for c in CONTROLS)
print("\npilot verdict:", "the test reads every control as it must" if passed else "a control is misread; the statistic or its margins are wrong")


stage 2: 200 draws per point, 10000 bootstrap resamples, 99% intervals



giant component: d_hat = 1.005
  below (d = 0.503): mean reached fraction by N 100: 0.02424, 1000: 0.00235, 10000: 0.00024
    ratio N=100 / N=10000 = 102.71 [78.94, 133.16]; slope of log fraction on log N -1.01; draws replaced {100: 302, 1000: 302, 10000: 298}
  above (d = 2.010): mean reached fraction by N 100: 0.73273, 1000: 0.73484, 10000: 0.72807
    ratio N=100 / N=10000 = 1.01 [0.93, 1.09]; slope of log fraction on log N -0.00; draws replaced {100: 24, 1000: 30, 10000: 30}
  reads: cliff; must read: cliff -> as it must



jump: d_hat = 2.253
  below (d = 1.126): mean reached fraction by N 100: 0.01657, 1000: 0.00166, 10000: 0.00017
    ratio N=100 / N=10000 = 100.39 [87.61, 115.95]; slope of log fraction on log N -1.00; draws replaced {100: 114, 1000: 83, 10000: 108}
  above (d = 4.505): mean reached fraction by N 100: 0.98874, 1000: 0.98809, 10000: 0.98832
    ratio N=100 / N=10000 = 1.00 [1.00, 1.00]; slope of log fraction on log N -0.00; draws replaced {100: 1, 1000: 1, 10000: 3}
  reads: cliff; must read: cliff -> as it must

smooth: d_hat = 0.250
  below (d = 0.125): mean reached fraction by N 100: 0.00803, 1000: 0.00851, 10000: 0.00852
    ratio N=100 / N=10000 = 0.94 [0.77, 1.12]; slope of log fraction on log N 0.01; draws replaced {100: 0, 1000: 0, 10000: 0}
  above (d = 0.500): mean reached fraction by N 100: 0.01727, 1000: 0.01827, 10000: 0.01802
    ratio N=100 / N=10000 = 0.96 [0.83, 1.10]; slope of log fraction on log N 0.01; draws replaced {100: 0, 1000: 0, 10000: 0}
  reads: no cliff; mu

## The Figure

`docs/figures/c18_cliff_controls.svg`: the mean reached fraction against N for the three controls, below and above each
one's located threshold, drawn here by matplotlib with a fixed `svg.hashsalt` and no date, so that drawing it again
changes no byte. The cell draws it twice and compares the two, then reports the file as *unchanged*, *changed* or *new*
against the one in `docs/figures/`; the check that matters is this notebook run again from a fresh kernel, then
`git diff --exit-code docs/figures`.


In [4]:
import hashlib
import io

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["svg.hashsalt"] = "c18"
plt.rcParams["svg.fonttype"] = "none"  # text stays text, as in the lab's diagrams: small, and it diffs
FIGURE = "docs/figures/c18_cliff_controls.svg"
STYLE = {"giant component": ("-", "o"), "jump": ("--", "s"), "smooth": (":", "^")}


def draw():
    """Two panels, the mean reached fraction against N below and above each control's located threshold, one ink."""
    fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.4), sharey=True)
    for ax, side, title in zip(axes, ("below", "above"), ("Below the located threshold, at d_hat / 2", "Above it, at 2 d_hat")):
        for control in CONTROLS:
            r = results[control]["rows"][side]
            line, marker = STYLE[control]
            ax.plot(SIZES, r["means"], linestyle=line, marker=marker, color="#1d1d1d", linewidth=1.5, markersize=5,
                    label=f"{control} (d = {r['d']:.2f})")
        if side == "below":
            guide = results["giant component"]["rows"]["below"]["means"][0] * 100 / np.array(SIZES)
            ax.plot(SIZES, guide, color="#9a9a9a", linewidth=1, label="1 / N guide")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xticks(SIZES, [str(n) for n in SIZES])
        ax.set_xlabel("N, units")
        ax.set_title(title, fontsize=10)
        ax.grid(True, which="major", color="#e4e4e4", linewidth=0.6)
        ax.legend(fontsize=7.5, frameon=False)
    axes[0].set_ylabel("mean reached fraction")
    fig.suptitle("C18's test on its three controls: finite reach below a threshold falls as 1 / N; above, it holds", fontsize=10)
    fig.tight_layout()
    out = io.StringIO()
    fig.savefig(out, format="svg", metadata={"Date": None})
    plt.close(fig)
    return out.getvalue()


first, second = draw(), draw()
assert first == second, "the figure differs between two drawings in one run"
before = open(FIGURE, encoding="utf-8").read() if os.path.exists(FIGURE) else None
status = "new" if before is None else ("unchanged" if before == first else "changed")
with open(FIGURE, "w", encoding="utf-8", newline="\n") as f:
    f.write(first)
print(f"{FIGURE}: {status}, {len(first.encode('utf-8'))} bytes, sha256 {hashlib.sha256(first.encode('utf-8')).hexdigest()}")


docs/figures/c18_cliff_controls.svg: unchanged, 45921 bytes, sha256 f1be4292b5149ebb8f005ad72b8e7350f07d3cf4ce48bd1ebf0bf0f139b92d6d
